In [0]:
%pip install gtfs-realtime-bindings requests
dbutils.library.restartPython()

In [0]:
import requests
import io
from databricks.sdk.runtime import *  # gives access to dbutils inside this context if needed

catalog = "transit_analytics"
landing_schema = "landing"
volume = "raw_gtfs"
base_url = "https://gtfs.adelaidemetro.com.au/v1"
feeds = ["vehicle_positions", "trip_updates"]

from datetime import datetime, timezone

for feed_name in feeds:
    url = f"{base_url}/realtime/{feed_name}"
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()

    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    file_path = f"/Volumes/{catalog}/{landing_schema}/{volume}/{feed_name}/{feed_name}_{timestamp}.pb"

    with open(file_path, "wb") as f:
        f.write(resp.content)

    print(f"Wrote {feed_name} -> {file_path}")